# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 01: Data Exploration
# ==========================================================

"""
Objective
---------
1. Load the SEER breast cancer dataset.
2. Verify dataset integrity.
3. Inspect dataset schema.
4. Perform initial exploratory data analysis (EDA).
5. Identify potential data quality issues before preprocessing.

Note
----
This notebook is ONLY for Exploratory Data Analysis (EDA).

No preprocessing will be performed here.
No missing values will be imputed.
No records will be removed.
No feature engineering will be applied.
"""

In [ ]:
# 1. Import Libraries
import os
import sys

from pyspark.sql import SparkSession

# Add project root directory
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions

from src.data.loader import (
    create_spark_session,
    load_csv
)

In [2]:
# 2. Create Spark Session

spark = create_spark_session(
    app_name="SEER Breast Cancer Survival Prediction"
)

print("Spark Session created successfully.")

Spark Session created successfully.


In [3]:
# 3. Load dataset

DATA_PATH = "../data/raw/breast_cancer_seer-2004-2015.csv"

df = load_csv(
    spark=spark,
    file_path=DATA_PATH
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [4]:
# 4. Dataset Overview

num_rows = df.count()
num_columns = len(df.columns)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print(f"Number of Rows    : {num_rows:,}")
print(f"Number of Columns : {num_columns}")

DATASET OVERVIEW
Number of Rows    : 457,351
Number of Columns : 29


In [5]:
# 5. Schema

print("=" * 60)
print("DATASET SCHEMA")
print("=" * 60)

df.printSchema()

DATASET SCHEMA
root
 |-- Age recode with <1 year olds and 90+: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race recode (W, B, AI, API): string (nullable = true)
 |-- Marital status at diagnosis: string (nullable = true)
 |-- CS tumor size (2004-2015): integer (nullable = true)
 |-- Survival months: string (nullable = true)
 |-- Vital status recode (study cutoff used): string (nullable = true)
 |-- Grade Recode (thru 2017): string (nullable = true)
 |-- PR Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- ER Status Recode Breast Cancer (1990+): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th T (1988-2015): string (nullable = true)
 |-- Breast - Adjusted AJCC 6th N (1988-2015): string (nullable = true)
 |-- Regional nodes examined (1988+): integer (nullable = true)
 |-- Regional nodes positive (1988+): integer (nullable = true)
 |-- Sequence number: string (nullable = true)
 |-- Patient ID: integer (nullable = true)
 |-- Primary Site: intege

In [6]:
# 6. Preview Dataset
df.show(10, truncate=False)

+------------------------------------+------+---------------------------+------------------------------+-------------------------+---------------+---------------------------------------+-----------------------------------+--------------------------------------+--------------------------------------+----------------------------------------+----------------------------------------+-------------------------------+-------------------------------+----------------+----------+------------+-----------------------+----------------------------+-------------------------+-----------------------+----------------------------------------+-------------------------------------------------+-----------------------------------+---------------------------------+-------------------------------------------------------------------------+------------------------------------+---------------------------------+--------------------------------------------+
|Age recode with <1 year olds and 90+|Sex   |Race recode (

In [26]:
# 7. Check Duplicate Records

print("=" * 60)
print("DUPLICATE RECORD ANALYSIS")
print("=" * 60)

total_rows = df.count()
unique_rows = df.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total Records      : {total_rows:,}")
print(f"Unique Records     : {unique_rows:,}")
print(f"Duplicate Records  : {duplicate_rows:,}")

if duplicate_rows == 0:
    print("\nNo duplicate records were found.")
else:
    print(f"\nWarning: {duplicate_rows:,} duplicate records detected.")

DUPLICATE RECORD ANALYSIS
Total Records      : 457,351
Unique Records     : 457,351
Duplicate Records  : 0

No duplicate records were found.


In [8]:
# 8. Check Missing Values (Bao nhiêu giá trị thiếu?)

from pyspark.sql.functions import col, when, count

print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

missing_df = df.select([
    count(
        when(
            col(c).isNull() | (col(c) == ""),
            c
        )
    ).alias(c)
    for c in df.columns
])

missing_counts = missing_df.collect()[0].asDict()

print(f"{'Column':45} {'Missing':>12}")

print("-" * 60)

for column, missing in missing_counts.items():
    print(f"{column[:45]:45} {missing:>12,}")

MISSING VALUE ANALYSIS
Column                                             Missing
------------------------------------------------------------
Age recode with <1 year olds and 90+                     0
Sex                                                      0
Race recode (W, B, AI, API)                              0
Marital status at diagnosis                              0
CS tumor size (2004-2015)                                0
Survival months                                          0
Vital status recode (study cutoff used)                  0
Grade Recode (thru 2017)                                 0
PR Status Recode Breast Cancer (1990+)                   0
ER Status Recode Breast Cancer (1990+)                   0
Breast - Adjusted AJCC 6th T (1988-2015)                 0
Breast - Adjusted AJCC 6th N (1988-2015)                 0
Regional nodes examined (1988+)                          0
Regional nodes positive (1988+)                          0
Sequence number                

In [9]:
# 9. Missing Value Percentage (Chiếm bao nhiêu %?)

print("=" * 60)
print("MISSING VALUE PERCENTAGE")
print("=" * 60)

for column, missing in missing_counts.items():

    percentage = (missing / total_rows) * 100

    print(
        f"{column[:45]:45} "
        f"{missing:>10,} "
        f"({percentage:6.2f}%)"
    )

MISSING VALUE PERCENTAGE
Age recode with <1 year olds and 90+                   0 (  0.00%)
Sex                                                    0 (  0.00%)
Race recode (W, B, AI, API)                            0 (  0.00%)
Marital status at diagnosis                            0 (  0.00%)
CS tumor size (2004-2015)                              0 (  0.00%)
Survival months                                        0 (  0.00%)
Vital status recode (study cutoff used)                0 (  0.00%)
Grade Recode (thru 2017)                               0 (  0.00%)
PR Status Recode Breast Cancer (1990+)                 0 (  0.00%)
ER Status Recode Breast Cancer (1990+)                 0 (  0.00%)
Breast - Adjusted AJCC 6th T (1988-2015)               0 (  0.00%)
Breast - Adjusted AJCC 6th N (1988-2015)               0 (  0.00%)
Regional nodes examined (1988+)                        0 (  0.00%)
Regional nodes positive (1988+)                        0 (  0.00%)
Sequence number                      

In [10]:
# 10. Data Type Analysis

print("=" * 60)
print("DATA TYPE ANALYSIS")
print("=" * 60)

print(f"{'Column Name':45} {'Spark Type'}")
print("-" * 60)

for column_name, data_type in df.dtypes:
    print(f"{column_name[:45]:45} {data_type}")

DATA TYPE ANALYSIS
Column Name                                   Spark Type
------------------------------------------------------------
Age recode with <1 year olds and 90+          string
Sex                                           string
Race recode (W, B, AI, API)                   string
Marital status at diagnosis                   string
CS tumor size (2004-2015)                     int
Survival months                               string
Vital status recode (study cutoff used)       string
Grade Recode (thru 2017)                      string
PR Status Recode Breast Cancer (1990+)        string
ER Status Recode Breast Cancer (1990+)        string
Breast - Adjusted AJCC 6th T (1988-2015)      string
Breast - Adjusted AJCC 6th N (1988-2015)      string
Regional nodes examined (1988+)               int
Regional nodes positive (1988+)               int
Sequence number                               string
Patient ID                                    int
Primary Site               

In [11]:
# 11. Variables Requiring Type Conversion

print("=" * 60)
print("VARIABLES THAT MAY REQUIRE TYPE CONVERSION")
print("=" * 60)

for column_name, data_type in df.dtypes:

    if data_type == "string":
        print(f"- {column_name}")

VARIABLES THAT MAY REQUIRE TYPE CONVERSION
- Age recode with <1 year olds and 90+
- Sex
- Race recode (W, B, AI, API)
- Marital status at diagnosis
- Survival months
- Vital status recode (study cutoff used)
- Grade Recode (thru 2017)
- PR Status Recode Breast Cancer (1990+)
- ER Status Recode Breast Cancer (1990+)
- Breast - Adjusted AJCC 6th T (1988-2015)
- Breast - Adjusted AJCC 6th N (1988-2015)
- Sequence number
- Behavior recode for analysis
- Laterality
- Diagnostic Confirmation
- Breast - Adjusted AJCC 6th M (1988-2015)
- Lymph-vascular Invasion (2004+ varying by schema)
- RX Summ--Surg Oth Reg/Dis (2003+)
- RX Summ--Surg/Rad Seq
- Radiation recode
- Chemotherapy recode (yes, no/unk)
- Breast - Adjusted AJCC 6th Stage (1988-2015)


In [12]:
# 12. Summary Statistics (Numerical Variables)

print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)

numerical_columns = [
    "CS tumor size (2004-2015)",
    "Regional nodes examined (1988+)",
    "Regional nodes positive (1988+)",
    "Patient ID",
    "Primary Site",
    "Histologic Type ICD-O-3",
    "RX Summ--Surg Prim Site (1998-2022)",
    "RX Summ--Surg Oth Reg/Dis (2003+)"
]

df.select(numerical_columns).describe().show(truncate=False)

SUMMARY STATISTICS
+-------+-------------------------+-------------------------------+-------------------------------+--------------------+------------------+-----------------------+-----------------------------------+------------------------------------------------------------+
|summary|CS tumor size (2004-2015)|Regional nodes examined (1988+)|Regional nodes positive (1988+)|Patient ID          |Primary Site      |Histologic Type ICD-O-3|RX Summ--Surg Prim Site (1998-2022)|RX Summ--Surg Oth Reg/Dis (2003+)                           |
+-------+-------------------------+-------------------------------+-------------------------------+--------------------+------------------+-----------------------+-----------------------------------+------------------------------------------------------------+
|count  |457351                   |457351                         |457351                         |457351              |457351            |457351                 |457351                             

In [13]:
# 13. Explore Categorical Variables

print("=" * 60)
print("CATEGORICAL VARIABLE ANALYSIS")
print("=" * 60)

categorical_columns = [
    "Sex",
    "Race recode (W, B, AI, API)",
    "Marital status at diagnosis",
    "Vital status recode (study cutoff used)",
    "Grade Recode (thru 2017)",
    "ER Status Recode Breast Cancer (1990+)",
    "PR Status Recode Breast Cancer (1990+)",
    "Breast - Adjusted AJCC 6th Stage (1988-2015)"
]

for column in categorical_columns:

    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)

    df.groupBy(column) \
      .count() \
      .orderBy("count", ascending=False) \
      .show(truncate=False)

CATEGORICAL VARIABLE ANALYSIS

Sex
+------+------+
|Sex   |count |
+------+------+
|Female|454233|
|Male  |3118  |
+------+------+


Race recode (W, B, AI, API)
+-----------------------------+------+
|Race recode (W, B, AI, API)  |count |
+-----------------------------+------+
|White                        |363658|
|Black                        |48333 |
|Asian or Pacific Islander    |39969 |
|American Indian/Alaska Native|2892  |
|Unknown                      |2499  |
+-----------------------------+------+


Marital status at diagnosis
+------------------------------+------+
|Marital status at diagnosis   |count |
+------------------------------+------+
|Married (including common law)|248525|
|Widowed                       |66827 |
|Single (never married)        |66391 |
|Divorced                      |47700 |
|Unknown                       |22144 |
|Separated                     |5032  |
|Unmarried or Domestic Partner |732   |
+------------------------------+------+


Vital status rec

In [14]:
# 14. Explore Numerical Variables

print("=" * 60)
print("NUMERICAL VARIABLE ANALYSIS")
print("=" * 60)

for column in numerical_columns:

    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)

    df.select(column).describe().show()

NUMERICAL VARIABLE ANALYSIS

CS tumor size (2004-2015)
+-------+-------------------------+
|summary|CS tumor size (2004-2015)|
+-------+-------------------------+
|  count|                   457351|
|   mean|       102.90888398626001|
| stddev|       266.67375590642416|
|    min|                        0|
|    max|                      999|
+-------+-------------------------+


Regional nodes examined (1988+)
+-------+-------------------------------+
|summary|Regional nodes examined (1988+)|
+-------+-------------------------------+
|  count|                         457351|
|   mean|              9.971678207766027|
| stddev|             19.850204978555343|
|    min|                              0|
|    max|                             99|
+-------+-------------------------------+


Regional nodes positive (1988+)
+-------+-------------------------------+
|summary|Regional nodes positive (1988+)|
+-------+-------------------------------+
|  count|                         457351|
|   mea

In [27]:
# ==========================================================
# 15. Detect SEER Special Codes
# ==========================================================

from pyspark.sql.functions import col

print("=" * 60)
print("SEER SPECIAL CODE DETECTION")
print("=" * 60)


# ----------------------------------------------------------
# Common SEER special codes
# ----------------------------------------------------------

special_string_codes = [
    "Unknown",
    "Unknown reason",
    "Blank(s)",
    "Not Applicable",
    "Unspecified",
    "Borderline"
]


special_numeric_codes = [
    999,
    998,
    997,
    996,
    995,
    99
]


special_code_summary = []


for column_name, dtype in df.dtypes:

    found_codes = []

    # ------------------------------------------------------
    # String columns
    # ------------------------------------------------------

    if dtype == "string":

        for code in special_string_codes:

            count = (
                df
                .filter(col(column_name) == code)
                .count()
            )

            if count > 0:

                found_codes.append(
                    f"{code} ({count:,})"
                )


    # ------------------------------------------------------
    # Numeric columns
    # ------------------------------------------------------

    elif dtype in [
        "int",
        "bigint",
        "double",
        "float"
    ]:

        for code in special_numeric_codes:

            count = (
                df
                .filter(col(column_name) == code)
                .count()
            )

            if count > 0:

                found_codes.append(
                    f"{code} ({count:,})"
                )


    # ------------------------------------------------------
    # Print Result
    # ------------------------------------------------------

    if found_codes:

        print("\n" + "-" * 60)
        print(column_name)
        print("-" * 60)

        for item in found_codes:

            print(item)

        special_code_summary.append(column_name)


print("\n")
print("=" * 60)
print("SPECIAL CODE SUMMARY")
print("=" * 60)

print(
    f"Variables containing special codes : {len(special_code_summary)}"
)

SEER SPECIAL CODE DETECTION



------------------------------------------------------------
Race recode (W, B, AI, API)
------------------------------------------------------------
Unknown (2,499)

------------------------------------------------------------
Marital status at diagnosis
------------------------------------------------------------
Unknown (22,144)

------------------------------------------------------------
CS tumor size (2004-2015)
------------------------------------------------------------
999 (27,129)
998 (2,102)
997 (81)
996 (140)
995 (649)
99 (59)

------------------------------------------------------------
Survival months
------------------------------------------------------------
Unknown (2,753)

------------------------------------------------------------
Grade Recode (thru 2017)
------------------------------------------------------------
Unknown (49,354)

------------------------------------------------------------
Regional nodes examined (1988+)
----------------------------------------

In [29]:
# ==========================================================
# 16. EDA Summary
# ==========================================================

print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)


print(f"Dataset Name        : SEER Breast Cancer")
print(f"Total Records       : {total_rows:,}")
print(f"Total Variables     : {len(df.columns)}")
print(f"Duplicate Records   : {duplicate_rows:,}")


# ----------------------------------------------------------
# Missing Values
# ----------------------------------------------------------

total_missing = sum(missing_counts.values())

columns_with_missing = sum(
    1 for value in missing_counts.values()
    if value > 0
)


print("\nMissing Values")
print("----------------")

print(
    f"Total Missing Values : {total_missing:,}"
)

print(
    f"Columns with Missing : {columns_with_missing}"
)


if total_missing > 0:

    print("\nColumns containing missing values:")

    for column, missing in missing_counts.items():

        if missing > 0:

            percentage = (
                missing / total_rows * 100
            )

            print(
                f"{column:35} "
                f"{missing:>10,} "
                f"({percentage:.2f}%)"
            )

else:

    print("No missing values detected.")



# ----------------------------------------------------------
# Data Types
# ----------------------------------------------------------

print("\nData Types")
print("----------------")


dtype_summary = {}

for _, dtype in df.dtypes:

    dtype_summary[dtype] = (
        dtype_summary.get(dtype, 0) + 1
    )


for dtype, count in dtype_summary.items():

    print(
        f"{dtype:10} : {count}"
    )



# ----------------------------------------------------------
# Data Quality Issues
# ----------------------------------------------------------

print("\nData Quality Issues")
print("----------------")


print(
    f"- Variables with special codes : "
    f"{len(special_code_summary)}"
)


if columns_with_missing > 0:

    print(
        "- Missing values require preprocessing"
    )


print(
    "- Variables requiring type conversion"
)


print(
    "- SEER-specific coding requires review"
)


print("\nEDA completed.")

EDA SUMMARY
Dataset Name        : SEER Breast Cancer
Total Records       : 457,351
Total Variables     : 29
Duplicate Records   : 0

Missing Values
----------------
Total Missing Values : 0
Columns with Missing : 0
No missing values detected.

Data Types
----------------
string     : 22
int        : 7

Data Quality Issues
----------------
- Variables with special codes : 10
- Variables requiring type conversion
- SEER-specific coding requires review

EDA completed.
